In [1]:
import math
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from torch import onnx


In [2]:
BOARD_SIZE = 5
K = 4  # four in a row


def has_four_in_a_row(board):
    # Rows
    for r in range(BOARD_SIZE):
        for c in range(BOARD_SIZE - K + 1):
            w = board[r, c:c+K]
            if abs(w.sum()) == K:
                return True

    # Columns
    for c in range(BOARD_SIZE):
        for r in range(BOARD_SIZE - K + 1):
            w = board[r:r+K, c]
            if abs(w.sum()) == K:
                return True

    # Diagonal \
    for r in range(BOARD_SIZE - K + 1):
        for c in range(BOARD_SIZE - K + 1):
            w = np.array([board[r+i, c+i] for i in range(K)])
            if abs(w.sum()) == K:
                return True

    # Diagonal /
    for r in range(BOARD_SIZE - K + 1):
        for c in range(K - 1, BOARD_SIZE):
            w = np.array([board[r+i, c-i] for i in range(K)])
            if abs(w.sum()) == K:
                return True

    return False

def action_to_index(row, col, symbol):
    sym = 0 if symbol == 1 else 1
    return row * 10 + col * 2 + sym

def index_to_action(idx):
    row = idx // 10
    col = (idx % 10) // 2
    symbol = 1 if idx % 2 == 0 else -1
    return row, col, symbol

def encode_state(board, current_player):
    x_plane = (board == 1).astype(np.float32)
    o_plane = (board == -1).astype(np.float32)
    empty_plane = (board == 0).astype(np.float32)
    turn_plane = np.ones_like(board, dtype=np.float32) \
        if current_player == 1 else np.zeros_like(board, dtype=np.float32)

    return np.stack([x_plane, o_plane, empty_plane, turn_plane], axis=0)



In [3]:
#test call

# Horizontal
b = np.zeros((5,5), dtype=int)
b[2, 0:4] = 1
assert has_four_in_a_row(b)

# Vertical
b = np.zeros((5,5), dtype=int)
b[0:4, 3] = -1
assert has_four_in_a_row(b)

# Diagonal \
b = np.zeros((5,5), dtype=int)
for i in range(4):
    b[i, i] = 1
assert has_four_in_a_row(b)

# Diagonal /
b = np.zeros((5,5), dtype=int)
for i in range(4):
    b[i, 4-i] = -1
assert has_four_in_a_row(b)

# Negative case
b = np.zeros((5,5), dtype=int)
b[0, 0:3] = 1
assert not has_four_in_a_row(b)

print("All tests passed")


All tests passed


In [4]:
class OrderChaosGame:
    def __init__(self):
        self.board = np.zeros((5, 5), dtype=np.int8)
        self.player = 1  # Order starts

    def clone(self):
        g = OrderChaosGame()
        g.board = self.board.copy()
        g.player = self.player
        return g

    def legal_actions(self):
        actions = []
        for r in range(5):
            for c in range(5):
                if self.board[r, c] == 0:
                    actions.append(action_to_index(r, c, 1))
                    actions.append(action_to_index(r, c, -1))
        return actions

    def apply(self, action_idx):
        r, c, s = index_to_action(action_idx)
        assert self.board[r, c] == 0
        self.board[r, c] = s
        self.player *= -1

    def terminal(self):
        if has_four_in_a_row(self.board):
            return True, 1  # Order wins when four in a row
        if np.all(self.board != 0):
            return True, -1  # Chaos wins on full board
        return False, 0

In [5]:
class MaxwellsDemonNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(4, 64, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 64, 3, padding=1)

        # Policy head
        self.policy_conv = nn.Conv2d(64, 2, 1)
        self.policy_fc = nn.Linear(2 * 5 * 5, 50)

        # Value head
        self.value_conv = nn.Conv2d(64, 1, 1)
        self.value_fc1 = nn.Linear(25, 64)
        self.value_fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))

        p = F.relu(self.policy_conv(x))
        p = p.view(p.size(0), -1)
        p = F.softmax(self.policy_fc(p), dim=1)

        v = F.relu(self.value_conv(x))
        v = v.view(v.size(0), -1)
        v = F.relu(self.value_fc1(v))
        v = torch.tanh(self.value_fc2(v))

        return p, v


In [6]:
class MCTSNode:
    def __init__(self, prior):
        self.P = prior          # prior probability
        self.N = 0              # visit count
        self.W = 0.0            # total value
        self.children = {}      # action_idx -> MCTSNode

    @property
    def Q(self):
        return 0 if self.N == 0 else self.W / self.N


In [ ]:
def puct(parent, child, c_puct=1.0):
    return child.Q + c_puct * child.P * math.sqrt(parent.N) / (1 + child.N)

def run_mcts(root_game: OrderChaosGame, net, simulations=100, c_puct=1.0):
    device = next(net.parameters()).device
    root = MCTSNode(prior=1.0)

    for _ in range(simulations):
        node = root
        game = root_game.clone()
        path = []

        # Selection
        while node.children:
            action, node = max(
                node.children.items(),
                key=lambda x: puct(path[-1] if path else root, x[1], c_puct)
            )
            game.apply(action)
            path.append(node)
        
        # Terminal check
        terminal, winner = game.terminal()
        if terminal:
            value = winner if game.player == 1 else -winner
        else:
            # Evaluation
            state = torch.tensor(
                encode_state(game.board, game.player),
                dtype=torch.float32
            ).unsqueeze(0).to(device)

            with torch.no_grad():
                policy, value = net(state)
                policy = policy.cpu().numpy()[0]
                value = value.item()

            # Expand
            legal = game.legal_actions()
            policy_masked = np.zeros(50)
            policy_masked[legal] = policy[legal]
            if policy_masked.sum() > 0:
                policy_masked /= policy_masked.sum()

            for a in legal:
                node.children[a] = MCTSNode(policy_masked[a])

        # Backup
        for n in reversed(path):
            n.N += 1
            n.W += value
            value = -value

        root.N += 1
        root.W += value

    # Improved policy
    pi = np.zeros(50)
    for a, child in root.children.items():
        pi[a] = child.N

    pi /= pi.sum()
    return pi


In [8]:
def self_play_game(net, simulations=100):
    game = OrderChaosGame()
    data = []

    while True:
        # Run MCTS from current position
        pi = run_mcts(game, net, simulations=simulations)

        # Store training example
        state = encode_state(game.board, game.player)
        data.append((state, pi, game.player))

        # Choose move (argmax for now)
        action = np.argmax(pi)
        game.apply(action)

        terminal, winner = game.terminal()
        if terminal:
            break

    # Assign final outcomes
    final_data = []
    for state, pi, player in data:
        z = winner if player == 1 else -winner
        final_data.append((state, pi, z))

    return final_data

def az_loss(p_pred, v_pred, pi_target, z_target):
    value_loss = F.mse_loss(v_pred.squeeze(), z_target)
    policy_loss = -(pi_target * torch.log(p_pred + 1e-8)).sum(dim=1).mean()
    return value_loss + policy_loss

def train_step(net, optimizer, batch):
    states, pis, zs = zip(*batch)

    states = torch.tensor(states, dtype=torch.float32)
    pis = torch.tensor(pis, dtype=torch.float32)
    zs = torch.tensor(zs, dtype=torch.float32)

    p_pred, v_pred = net(states)
    loss = az_loss(p_pred, v_pred, pis, zs)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()



In [ ]:
#Training loop

MAX_BUFFER_SIZE = 3500
replay_buffer = []
NUM_ITERATIONS = 200
SELF_PLAY_GAMES = 15
SIMULATIONS_PER_MOVE = 150
BATCH_SIZE = 64

net = MaxwellsDemonNet()
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)

for iteration in range(NUM_ITERATIONS):
    print(f"\n=== Iteration {iteration} ===")

    # --- Self-play ---
    net.eval()
    for g in range(SELF_PLAY_GAMES):
        game_data = self_play_game(net, simulations=SIMULATIONS_PER_MOVE)
        replay_buffer.extend(game_data)

    # Trim buffer
    if len(replay_buffer) > MAX_BUFFER_SIZE:
        replay_buffer = replay_buffer[-MAX_BUFFER_SIZE:]

    print(f"Replay buffer size: {len(replay_buffer)}")

    # --- Training ---
    net.train()
    losses = []

    for _ in range(20):  # 20 gradient steps per iteration
        batch = random.sample(
            replay_buffer,
            min(BATCH_SIZE, len(replay_buffer))
        )
        loss = train_step(net, optimizer, batch)
        losses.append(loss)

    print(f"Mean loss: {sum(losses)/len(losses):.4f}")




=== Iteration 0 ===
Replay buffer size: 120
Mean loss: 4.4867

=== Iteration 1 ===
Replay buffer size: 270
Mean loss: 3.7947

=== Iteration 2 ===
Replay buffer size: 630
Mean loss: 3.6397

=== Iteration 3 ===
Replay buffer size: 795
Mean loss: 3.4381

=== Iteration 4 ===
Replay buffer size: 990
Mean loss: 3.3061

=== Iteration 5 ===
Replay buffer size: 1230
Mean loss: 3.2357

=== Iteration 6 ===
Replay buffer size: 1470
Mean loss: 3.1043

=== Iteration 7 ===
Replay buffer size: 1740
Mean loss: 2.9997

=== Iteration 8 ===
Replay buffer size: 1920
Mean loss: 2.8942

=== Iteration 9 ===
Replay buffer size: 2070
Mean loss: 2.7942

=== Iteration 10 ===
Replay buffer size: 2310
Mean loss: 2.7285

=== Iteration 11 ===
Replay buffer size: 2445
Mean loss: 2.7054

=== Iteration 12 ===
Replay buffer size: 2715
Mean loss: 2.6832

=== Iteration 13 ===
Replay buffer size: 2895
Mean loss: 2.6467

=== Iteration 14 ===
Replay buffer size: 3120
Mean loss: 2.6215

=== Iteration 15 ===
Replay buffer size

In [10]:
#test model outputs for an array of inputs of different board states

net.eval()
test_states = [
    encode_state(np.zeros((5, 5), dtype=np.int8), 1),  # Empty board, Order's turn
    encode_state(np.array([[1, -1, 0, 0, 0],
                           [0, 1, -1, 0, 0],
                           [0, 0, 1, -1, 0],
                           [0, 0, 0, 0, 0],
                           [0, 0, 0, 0, 0]], dtype=np.int8), 1),  # Near win
    encode_state(np.array([[0, 1, 1, 1, 0],
                           [-1, -1, -1, 0, 0],
                           [0, 0, 0, 0, 0],
                           [0, 0, 0, 0, 0],
                           [0, 0, 0, 0, 0]], dtype=np.int8), -1),  # Near loss
    # near winning situation for chaos, there need to be an odd number of moves playes as well
    encode_state(np.array([[ 1, 1, 1, 0,-1],
                           [-1,-1, -1, 1, 0],
                           [ 0, 1,-1, -1, 0],
                           [ 0, 0, 1, 1, 0],
                           [ 0, -1, 0, 1, 0]], dtype=np.int8), -1)
]
for i, state in enumerate(test_states):
    state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        policy, value = net(state_tensor)
    print(f"Test State {i}:")
    print(f"best move: {index_to_action(np.argmax(policy.cpu().numpy()))}")
    print(f"Value: {value.item()}\n")

Test State 0:
best move: (3, 1, -1)
Value: 0.9999994039535522

Test State 1:
best move: (4, 1, 1)
Value: 0.9999997019767761

Test State 2:
best move: (2, 0, -1)
Value: -1.0

Test State 3:
best move: (4, 0, 1)
Value: -1.0



In [11]:
# Export to ONNX
net.eval()
net.cpu()

output_path = "../frontend/public/maxwells_demon.onnx"

dummy_input = torch.zeros(
    1, 4, 5, 5,
    dtype=torch.float32
)

torch.onnx.export(
    net,
    dummy_input,
    output_path,
    input_names=["board"],
    output_names=["policy", "value"],
    dynamic_axes={
        "board": {0: "batch"},
        "policy": {0: "batch"},
        "value": {0: "batch"}
    },
    opset_version=17
)

